# V23: pretrained proposal refinement and five-fold medoid consensus

This notebook runs the complete image-to-submission inference pipeline using the ten included checkpoints. It does not train a model or read a cached submission. The final architecture uses V3 refiners for folds 0–2 and V2 refiners for folds 3–4, with one YOLO11s detector per fold.

The historical public score is 0.39 and the archived strict-OOF reference is 0.43520. These are historical results. A newly generated full CSV is checked against the historical SHA-256 and its actual environment is recorded.

Obtain the official `filament-segmentation-2026` test images through your own Kaggle account after accepting the competition rules. This repository does not redistribute those images or the annotations. Run this notebook from the checked-out repository. The historical device configuration was **two Tesla T4 GPUs**.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path.cwd().resolve()  # Change this if the notebook server started elsewhere.
assert (REPO / "inference.py").is_file(), "Set REPO to the repository root."
assert (REPO / "weights" / "manifest.json").is_file()
print(REPO)


## Install the declared inference target

`requirements.txt` pins the direct numerical and model dependencies. Python 3.12.13 is the historical interpreter. The full historical environment was not saved; the README distinguishes observed versions from the declared porting target. Installation may require an Internet connection. Restart the notebook kernel if package installation changes an already imported package.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)


## Verify the model files and saved architecture

The first check needs only the Python standard library. The second checks strict checkpoint loading and synthetic output shapes without opening competition images.


In [ ]:
subprocess.run([sys.executable, str(REPO / "inference.py"), "--verify-only"], check=True)
subprocess.run([sys.executable, str(REPO / "scripts" / "verify_models.py")], cwd=REPO, check=True)


## Select the official test images

Set `IMAGE_DIR` to the folder containing the 180 official test JPEGs. On Kaggle, the discovery below locates an attached official `test/test_images` folder. No training labels are needed for inference.


In [ ]:
IMAGE_DIR = None  # Example: Path("/data/MAGFiLO_1.0_Kaggle_2026/test/test_images")
if IMAGE_DIR is None:
    candidates = sorted(Path("/kaggle/input").glob("**/test/test_images")) if Path("/kaggle/input").exists() else []
    if len(candidates) != 1:
        raise RuntimeError("Set IMAGE_DIR to the official test/test_images directory.")
    IMAGE_DIR = candidates[0]
IMAGE_DIR = Path(IMAGE_DIR)
assert len(list(IMAGE_DIR.glob("*.jpeg"))) == 180
print(IMAGE_DIR)


## Run the complete pretrained ensemble

The script loads the five detector checkpoints and the five selected refiners, predicts from every JPEG, applies flip averaging and fold consensus, and writes compressed COCO RLE. It requires exactly two CUDA devices and does not fall back to another device. The original V23 run used two T4s; another GPU or numerical-library version may change threshold-adjacent pixels.

For a deliberate remote smoke run, add `--limit-images 2` and remove `--check-reference`; the resulting run is marked `smoke_only` and cannot establish full reproduction.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/v23_reproduction") if Path("/kaggle/working").exists() else REPO / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "submission.csv"
command = [sys.executable, str(REPO / "inference.py"),
           "--images", str(IMAGE_DIR), "--weights", str(REPO / "weights"),
           "--output", str(OUTPUT_CSV), "--device", "cuda", "--require-gpus", "2",
           "--check-reference"]
subprocess.run(command, cwd=REPO, check=True)


## Inspect the result and provenance

A passing reference check means that all output bytes match the historical V23 CSV. The JSON records the actual versions, GPU names, checkpoint-manifest hash, source hashes, image count and fallback count. A mismatch is retained in that JSON for investigation; it is not reported as exact reproduction.


In [ ]:
run = json.loads(OUTPUT_CSV.with_suffix(".run.json").read_text())
print(json.dumps(run, indent=2))
assert run["images"] == 180
assert run["matches_historical_reference"] is True
import pandas as pd
submission = pd.read_csv(OUTPUT_CSV)
print(submission.shape)
display(submission.head())


The final file is `submission.csv`. Attribution, historical evidence and validation limits are documented in `README.md`, `ATTRIBUTION.md` and `provenance/`. No CSV is submitted to Kaggle by this notebook.
